# <font color="#418FDE" size="6.5" uppercase>**Pipeline Construction**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Construct pipelines that chain preprocessing and estimators safely. 
- Access, slice, and modify pipeline steps and nested parameters. 
- Use caching and visualization to inspect pipeline behavior. 


## **1. Pipeline Basics**

### **1.1. Chaining Steps**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_06/Lecture_A/image_01_01.jpg?v=1783984831" width="250">



>* Chain preprocessing before the final estimator
>* Create one reproducible modeling workflow

>* Fit preprocessing only on training data
>* Prevent leakage for honest model evaluation

>* Transformers prepare data; estimators produce outputs
>* Pipelines keep workflows consistent and reliable



In [ ]:
#@title Python Code - Chaining Steps

# This example chains preprocessing and modeling steps.
# A pipeline prevents leakage during training.
# The output compares safe and unsafe workflows.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged classification dataset.
data = load_breast_cancer()
features = data.data
target = data.target

# Check that features and labels match.
if features.shape[0] != target.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split before fitting any preprocessing step.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.25,
    stratify=target,
    random_state=42,
)

# Chain scaling and logistic regression into one estimator.
safe_pipeline = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

# Fit the whole chain only on training data.
safe_pipeline.fit(X_train, y_train)
safe_predictions = safe_pipeline.predict(X_test)
safe_accuracy = accuracy_score(y_test, safe_predictions)

# This unsafe scaler learns from all rows before splitting.
unsafe_scaler = StandardScaler()
scaled_features = unsafe_scaler.fit_transform(features)

# The same split is now made after leaked preprocessing.
X_train_bad, X_test_bad, y_train_bad, y_test_bad = train_test_split(
    scaled_features,
    target,
    test_size=0.25,
    stratify=target,
    random_state=42,
)

# Fit only the model after preprocessing has already leaked.
unsafe_model = LogisticRegression(max_iter=1000, random_state=42)
unsafe_model.fit(X_train_bad, y_train_bad)
unsafe_predictions = unsafe_model.predict(X_test_bad)
unsafe_accuracy = accuracy_score(y_test_bad, unsafe_predictions)

print("scikit-learn version:", sklearn.__version__)
print("Pipeline steps:", list(safe_pipeline.named_steps.keys()))
print("Safe pipeline accuracy:", round(safe_accuracy, 3))
print("Unsafe pre-split scaling accuracy:", round(unsafe_accuracy, 3))
print("Key idea: fit preprocessing inside the pipeline.")



### **1.2. Pipeline Constructor**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_06/Lecture_A/image_01_02.jpg?v=1783984835" width="250">



>* Name and order preprocessing-to-model steps
>* Pass outputs forward to avoid mistakes

>* Prevent leakage during evaluation and deployment
>* Fit preprocessing only on training data

>* Transformers first, task estimator last
>* Named pipeline supports reuse and collaboration



In [ ]:
#@title Python Code - Pipeline Constructor

# Build a safe preprocessing and modeling pipeline.
# Chain scaling before logistic regression automatically.
# Inspect named steps and compare test accuracy.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged classification dataset.
data = load_breast_cancer()
features = data.data
target = data.target

# Validate the basic shape before modeling.
if features.shape[0] != target.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split before fitting to avoid data leakage.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.25, stratify=target, random_state=42
)

# Name each pipeline step in execution order.
pipe = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

# Fit calls scaler.fit_transform, then model.fit.
pipe.fit(X_train, y_train)
predictions = pipe.predict(X_test)

# Access steps and nested parameters by name.
step_names = list(pipe.named_steps.keys())
model_c = pipe.get_params()["model__C"]

# Print concise results that reveal pipeline behavior.
print("scikit-learn version:", sklearn.__version__)
print("Pipeline steps:", step_names)
print("Nested parameter model__C:", model_c)
print("Test accuracy:", round(accuracy_score(y_test, predictions), 3))



### **1.3. Automatic Pipeline Creation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_06/Lecture_A/image_01_03.jpg?v=1783984833" width="250">



>* Automatically chain preprocessing before model training
>* Reduce setup with sensible step names

>* Fit preprocessing only on training data
>* Prevent leakage for trustworthy deployment

>* Quickly test transformer and estimator combinations
>* Design steps carefully for reliable pipelines



In [ ]:
#@title Python Code - Automatic Pipeline Creation

# Build an automatic pipeline for safe modeling.
# Make_pipeline names steps and preserves order.
# Accuracy confirms the combined workflow runs correctly.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged classification dataset.
data = load_breast_cancer()
features = data.data
target = data.target

# Validate the basic shape before modeling.
if features.shape[0] != target.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split before fitting to avoid data leakage.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.25,
    random_state=42,
    stratify=target,
)

# Make_pipeline automatically creates safe step names.
pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42),
)

# Fit preprocessing and model together on training data.
pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

# Show the automatically assigned step names.
step_names = list(pipeline.named_steps.keys())
print("scikit-learn version:", sklearn.__version__)
print("Automatic step names:", step_names)
print("Test accuracy:", round(accuracy, 3))
print("Final estimator step:", step_names[-1])



## **2. Pipeline Step Control**

### **2.1. Access Pipeline Steps**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_06/Lecture_A/image_02_01.jpg?v=1783984826" width="250">



>* Inspect each named pipeline operation
>* Ensure consistent preprocessing for predictions

>* Access steps by position or name
>* Inspect fitted transformers and final estimators

>* Slice pipelines to inspect and debug workflows
>* Audit components for transparent, maintainable models



In [ ]:
#@title Python Code - Access Pipeline Steps

# This example accesses named pipeline steps.
# It shows fitted transformers and estimators.
# The output confirms safe step inspection.

import sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Load a small packaged classification dataset.
iris = load_iris()
X = iris.data
y = iris.target

# Validate the basic shape before modeling.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split data so preprocessing is fitted only on training rows.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Build a two-step pipeline with clear step names.
pipe = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=200, random_state=42)),
    ]
)

# Fit the whole pipeline once.
pipe.fit(X_train, y_train)

# Access steps by name and by position.
scaler_by_name = pipe.named_steps["scaler"]
final_by_position = pipe[-1]

# Slice the pipeline to keep only preprocessing.
preprocessing_only = pipe[:-1]
first_test_row_scaled = preprocessing_only.transform(X_test[:1])

# Print concise evidence of each access method.
print("scikit-learn version:", sklearn.__version__)
print("Step names:", list(pipe.named_steps.keys()))
print("Named scaler mean, first 3:", scaler_by_name.mean_[:3].round(2).tolist())
print("Final step by position:", type(final_by_position).__name__)
print("Scaled first test row, first 3:", first_test_row_scaled[0, :3].round(2).tolist())
print("Test accuracy:", round(pipe.score(X_test, y_test), 3))



### **2.2. Nested Parameter Tuning**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_06/Lecture_A/image_02_02.jpg?v=1783984828" width="250">



>* Tune component settings through named pipeline steps
>* Keep preprocessing and modeling connected safely

>* Step names target the correct inner setting
>* Stored configurations improve tuning reproducibility

>* Tune preprocessing and models together
>* Compare configurations safely within one pipeline



In [ ]:
#@title Python Code - Nested Parameter Tuning

# This script tunes nested pipeline parameters.
# Double underscores target settings inside steps.
# Printed scores show each controlled change.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Validate the basic shape before modeling.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split data before fitting the pipeline.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Build one pipeline with named steps.
pipe = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        ("select", SelectKBest(score_func=f_classif, k=10)),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

# Fit and score the starting configuration.
pipe.fit(X_train, y_train)
baseline_accuracy = accuracy_score(y_test, pipe.predict(X_test))

# Change inner settings through the outer pipeline.
pipe.set_params(select__k=5, model__C=0.2)
pipe.fit(X_train, y_train)
tuned_accuracy = accuracy_score(y_test, pipe.predict(X_test))

# Read nested settings back from the pipeline.
current_k = pipe.get_params()["select__k"]
current_c = pipe.get_params()["model__C"]

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Baseline accuracy with select__k=10: {baseline_accuracy:.3f}")
print(f"Tuned accuracy with select__k={current_k}: {tuned_accuracy:.3f}")
print(f"Nested model__C is now: {current_c}")



### **2.3. Passthrough and Drop**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_06/Lecture_A/image_02_03.jpg?v=1783984830" width="250">



>* Passthrough keeps data unchanged in the pipeline
>* Drop removes steps to compare preprocessing choices

>* Passthrough keeps features, skipping transformation
>* Drop removes selected feature groups entirely

>* Adjust steps for controlled model experiments
>* Keep pipelines consistent, inspectable, and flexible



In [ ]:
#@title Python Code - Passthrough and Drop

# This example compares passthrough and drop controls.
# It modifies nested pipeline steps by name.
# The printed scores show each setting clearly.

import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged regression dataset.
diabetes = load_diabetes(as_frame=True)
X = diabetes.data[["bmi", "bp", "s1", "s5"]]
y = diabetes.target

# Validate the small teaching dataset shape.
if X.shape[1] != 4 or len(y) != len(X):
    raise ValueError("Expected four features and matching target values.")

# Split before fitting to avoid data leakage.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Build one pipeline with two named preprocessing branches.
preprocess = ColumnTransformer(
    [("scale_bmi_bp", StandardScaler(), ["bmi", "bp"]),
     ("use_s1_s5", "passthrough", ["s1", "s5"])]
)

# The model step stays unchanged across experiments.
pipe = Pipeline(
    [("preprocess", preprocess), ("model", LinearRegression())]
)

# Fit the original pipeline and score it.
pipe.fit(X_train, y_train)
base_score = r2_score(y_test, pipe.predict(X_test))

# Passthrough keeps bmi and bp, but skips scaling them.
pipe.set_params(preprocess__scale_bmi_bp="passthrough")
pipe.fit(X_train, y_train)
pass_score = r2_score(y_test, pipe.predict(X_test))

# Drop removes bmi and bp from the model input entirely.
pipe.set_params(preprocess__scale_bmi_bp="drop")
pipe.fit(X_train, y_train)
drop_score = r2_score(y_test, pipe.predict(X_test))

# Print concise evidence of the changed nested step.
print("scikit-learn version:", sklearn.__version__)
print("Original scaled branch R2:", round(base_score, 3))
print("Passthrough branch R2:", round(pass_score, 3))
print("Dropped branch R2:", round(drop_score, 3))
print("Current nested setting:", pipe.get_params()["preprocess__scale_bmi_bp"])



## **3. Pipeline Utilities**

### **3.1. Pipeline Slicing**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_06/Lecture_A/image_03_01.jpg?v=1783984820" width="250">



>* Inspect selected preprocessing steps before prediction
>* Verify transformations and prevent data leakage

>* Isolate pipeline stages to find errors
>* Explain data transformations to collaborators

>* Inspect intermediate stages and choose caching points
>* Connect diagrams to transformed data views



In [ ]:
#@title Python Code - Pipeline Slicing

# This script demonstrates slicing a fitted pipeline.
# Slicing reveals intermediate transformed feature values.
# The output compares raw and transformed shapes.

import sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Load a small packaged classification dataset.
iris = load_iris()
X = iris.data
y = iris.target

# Check the dataset has matching rows and labels.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows and labels must match.")

# Split before fitting to avoid data leakage.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Build one pipeline with preprocessing and a classifier.
pipe = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        ("reduce", PCA(n_components=2, random_state=42)),
        ("model", LogisticRegression(max_iter=200, random_state=42)),
    ]
)

# Fit the complete pipeline once.
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

# Slice away the final estimator to inspect preprocessing only.
preprocess_only = pipe[:-1]
X_test_prepared = preprocess_only.transform(X_test)

# Slice the first step to inspect scaling before PCA.
scaling_only = pipe[:1]
X_test_scaled = scaling_only.transform(X_test)

print("scikit-learn version:", sklearn.__version__)
print("Full pipeline steps:", list(pipe.named_steps.keys()))
print("Preprocessing slice steps:", list(preprocess_only.named_steps.keys()))
print("Raw test shape:", X_test.shape)
print("After scaling shape:", X_test_scaled.shape)
print("After scaling and PCA shape:", X_test_prepared.shape)
print("First prepared row:", X_test_prepared[0].round(3).tolist())
print("Test accuracy:", round(accuracy, 3))



### **3.2. Caching Transforms**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_06/Lecture_A/image_03_02.jpg?v=1783984822" width="250">



>* Store costly preprocessing results for reuse
>* Speed up repeated model experiments

>* Reuse unchanged preprocessing during model tuning
>* Best for costly transforms, not simple ones

>* See when pipeline stages recompute
>* Identify stable steps and changing work



In [ ]:
#@title Python Code - Caching Transforms

# This example shows pipeline transform caching.
# A custom transformer counts repeated fitting work.
# The second fit reuses cached preprocessing results.

import tempfile

import numpy as np
import sklearn
from joblib import Memory
from sklearn.base import BaseEstimator
from sklearn.base import TransformerMixin
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# This transformer records how often fitting really happens.
class CountingScaler(BaseEstimator, TransformerMixin):
    fit_count = 0

    def fit(self, X, y=None):
        CountingScaler.fit_count = CountingScaler.fit_count + 1
        self.scaler_ = StandardScaler()
        self.scaler_.fit(X, y)
        return self

    def transform(self, X):
        return self.scaler_.transform(X)

# Load a small packaged classification dataset.
data = load_breast_cancer()
selected = np.r_[
    np.flatnonzero(data.target == 0)[:60],
    np.flatnonzero(data.target == 1)[:60],
]
X = data.data[selected]
y = data.target[selected]

# Validate the basic shape before building the pipeline.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target labels.")

with tempfile.TemporaryDirectory() as cache_dir:
    # Use a temporary joblib cache object for this demonstration.
    cache = Memory(location=cache_dir, verbose=0)

    # The final estimator changes, but preprocessing stays identical.
    first_pipeline = Pipeline(
        steps=[("scale", CountingScaler()), ("model", LogisticRegression(max_iter=500))],
        memory=cache,
    )

    second_pipeline = Pipeline(
        steps=[("scale", CountingScaler()), ("model", LogisticRegression(C=0.5, max_iter=500))],
        memory=cache,
    )

    # Fit once, then fit again with the same cached transform step.
    CountingScaler.fit_count = 0
    first_pipeline.fit(X, y)
    first_count = CountingScaler.fit_count

    second_pipeline.fit(X, y)
    second_count = CountingScaler.fit_count

# Print concise evidence that caching avoided repeated preprocessing.
print("scikit-learn version:", sklearn.__version__)
print("Scaler fits after first pipeline:", first_count)
print("Scaler fits after second pipeline:", second_count)
print("Cached reuse happened:", second_count == first_count)



### **3.3. Pipeline Diagrams**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_06/Lecture_A/image_03_03.jpg?v=1783984824" width="250">



>* Visualize pipeline flow and step order
>* Verify preprocessing logic for safer modeling

>* Show complex branches and feature routing
>* Spot pipeline design mistakes early

>* Share pipeline flow clearly across teams
>* Spot errors, costs, and reusable steps



In [ ]:
#@title Python Code - Pipeline Diagrams

# This script displays a scikit-learn pipeline diagram.
# The diagram reveals preprocessing and estimator structure.
# Printed checks connect diagram inspection to behavior.

import sklearn
from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged dataset for a fast example.
iris = load_iris(as_frame=True)
features = iris.data
labels = iris.target

# Validate the basic shape before building the pipeline.
if features.shape[0] != labels.shape[0]:
    raise ValueError("Feature rows must match target length.")

# Route two feature groups through separate preprocessing branches.
sepal_columns = ["sepal length (cm)", "sepal width (cm)"]
petal_columns = ["petal length (cm)", "petal width (cm)"]

preprocessor = ColumnTransformer(
    transformers=[
        ("sepal_scale", StandardScaler(), sepal_columns),
        ("petal_scale", StandardScaler(), petal_columns),
    ]
)

# Chain preprocessing into one simple classifier.
pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", LogisticRegression(max_iter=200, random_state=42)),
    ]
)

# Fit once so the diagram represents a trained workflow.
pipeline.fit(features, labels)
accuracy = pipeline.score(features, labels)

# Enable the notebook-friendly visual diagram representation.
set_config(display="diagram")

print("scikit-learn version:", sklearn.__version__)
print("Pipeline steps:", list(pipeline.named_steps.keys()))
print("Preprocessing branches:", [name for name, _, _ in preprocessor.transformers])
print("Training accuracy:", round(accuracy, 3))
print("In Colab, display(pipeline) shows the interactive pipeline diagram.")
display(pipeline)



# <font color="#418FDE" size="6.5" uppercase>**Pipeline Construction**</font>


In this lecture, you learned to:
- Construct pipelines that chain preprocessing and estimators safely. 
- Access, slice, and modify pipeline steps and nested parameters. 
- Use caching and visualization to inspect pipeline behavior. 

In the next Lecture (Lecture B), we will go over 'Feature Spaces'